# Pedal floor sandwich panel: load-derivation sweep

A stiffness model of the panel with strength checks of the core and hardpoint (the face sheets are not checked). For each row of `inputs/sweep.csv`, and for every pedal box position along the left slot:

1. **Geometry** (PyAnsys Geometry / SpaceClaim): trapezoid panel with both slots cut out, the hardpoint imprinted around the left slot only (trimmed at the panel edges if `hardpoint_rad` reaches past them), the pedal box STEP surface placed on the slot with bolt-head patches (`washer_diameter_top`), and a separate washer body under each nut on the panel's back face (`washer_diameter_bottom`, `washer_thickness`, `washer_material`).
2. **Mesh** (PyMechanical): coarse or fine (`mesh_quality`), refined on the hardpoint, slots and washers.
3. **Layup** (PyACP): face plies everywhere; core everywhere except the hardpoint, where the hardpoint material takes its place.
4. **Solve** (PyMechanical): frictionless contact pedal box to panel, rough contact washers to panel, two grade 8 beam bolts from the nut footprint on each washer (`nut_bearing_diameter`) to the head patch on the pedal box, front/left/right edges supported (rear free).
   - Load step 1: bolt pretension. Load step 2: bolts locked, pedal load applied.
5. **Results**: bolt reaction forces, net out-of-plane force per joint, contact pressures, deflection, strength margins, panel mass and runtime per run.

The stage code is in `pedal_floor/`; this notebook only drives it.

### Axes
Origin at the middle of the rear edge, **X** toward the front edge, **Y** to the right, **Z** into the page (as in the sketch). The panel's back (nut-side) face lies in z = 0 and the laminate builds toward -Z; the pedal box sits on the -Z side against the face at z = -panel thickness. Ply angle 0° is global X.

### Inputs (`inputs/`), all with units
- **`constants.csv`**: `parameter,value,unit,description`, shared by every run. Units: `in`, `mm`, `lbf`, `N`, `psi`, `ksi`, `MPa`, `Pa`, `GPa`, `pcf`, `kg/m3`, `deg`, `-`.
- **`sweep.csv`**: one row per case; units go in the header, e.g. `hardpoint_rad [in]`. Any column overrides the constant of the same name, so a parameter can move between the files without code changes.
- **`materials.csv`**: `material,property,value,unit,note`, one property per row like an Engineering Data card (E1..E3, nu12/nu23/nu13, G12/G23/G13, density, ply_type, ply_thickness for face plies; E and nu for isotropic). Strengths for the checks: `shear_strength_L/W`, `compressive_strength`, `yield_strength`, each with `_min` and/or `_typ`.
- **`pedalbox_cad`** (STEP surface): its rear hole at the CAD origin; the two holes are detected automatically.

When editing the CSVs in Excel, avoid text values that start with `+` or `-` (Excel turns them into formulas).

**Layup syntax**: `(0 U U U U U U U U U U U U 45) Core (45 U U U U U U U U U U U U 0)`. `U` = UD ply at 0°, a number = woven ply at that angle, `Core` = the core/hardpoint layer. Parentheses are only for readability. Listed from the panel face against the pedal box outward.

**Pedal box positions**: from the rearmost (rear bolt at the rear end of the slot) forward in steps of `pedalbox_offset_step` to the foremost (front bolt at the front end of the slot). A step of 0 runs only the rearmost position.

### Outputs (`outputs/`)
- `results.csv`: one row per case and position, rewritten after each run. Units set by `result_units`. Strength checks give stress, allowable, margin (`_MS`) and pass/fail (`_ok`), then `core_ok`, `hardpoint_ok`, `passes` (this run) and `case_passes` (all positions of the case).
- `case_###_offX.XXin/`: `model.mechdb` (open in Mechanical), `acp_model.acph5` (open in ACP), `images/` (a plan view of every result in the Solution tree), `params.json`, `log.txt`.

### Results viewer (`results_viewer/`)
- `results_viewer.html`: every run's plots and `results.csv` on one page with filters (see `results_viewer.ipynb`); rewritten after every run. The folder keeps a copy of each run's plots and results, so deleting run folders from `outputs/` does not remove them from the viewer.

In [ ]:
# Re-import pedal_floor automatically when its files change, so code updates
# take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import logging

import pandas as pd
from IPython.display import Image, display

from pedal_floor import config, runner

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("ansys").setLevel(logging.WARNING)

display(pd.read_csv(config.INPUTS / "constants.csv"))
display(pd.read_csv(config.INPUTS / "sweep.csv"))

Check every case before starting Ansys (geometry fits, materials exist, layup parses).

In [ ]:
constants, materials = config.load_constants(), config.load_materials()
for row in config.load_sweep():
    try:
        p = config.case_params(constants, row, materials)
        print(f"case {p['case_id']}: ok, panel {config.panel_thickness(p, materials) / 25.4:.3f} in thick, {len(config.parse_layup(p['layup'], p)) - 1} plies")
    except ValueError as exc:
        print(f"case {row['case_id']}: {exc}")

Optional: write a full-factorial sweep to `inputs/sweep_full_factorial.csv` (it does not touch `sweep.csv`; rename it if you want to use it). Each case runs at every pedal box position, so the run count is cases × positions.

In [ ]:
import itertools

layup = "(0 U U U U U U U U U U U U 45) Core (45 U U U U U U U U U U U U 0)"
levels = {
    "hardpoint_rad [in]": [0.25, 0.375, 0.5, 0.75],
    "hardpoint_material": ["Al7075_T6", "Balsa_150"],
    "washer_diameter_bottom [in]": [0.5],
    "core_material": ["Honeycomb_3.1pcf", "Honeycomb_4.5pcf", "Honeycomb_6.1pcf", "Honeycomb_8.1pcf"],
    "core_thickness [in]": [0.5],
    "layup": [layup],
    "edge_bc": ["fixed", "simply_supported"],
}
factorial = pd.DataFrame(list(itertools.product(*levels.values())), columns=list(levels))
factorial.insert(0, "case_id", range(1, len(factorial) + 1))
factorial.to_csv(config.INPUTS / "sweep_full_factorial.csv", index=False)
print(f"{len(factorial)} cases written")

Start SpaceClaim and ACP once; they are reused for every run. Mechanical starts fresh for each run (it carries state between models in one session). Set `mechanical_ui=True` to watch Mechanical build each model.

In [ ]:
sessions = runner.Sessions.start(mechanical_ui=False)

Run one case at one position first and check it, then the whole sweep. `offsets_in` picks positions (in inches from the rearmost); leave it out for all of them. Re-running a case at a position replaces its row in `results.csv`.

Rough runtimes: about 170 s per run on the coarse mesh; the fine mesh takes longer.

In [ ]:
first_case = int(config.load_sweep()[0]["case_id"])
results = runner.run_sweep(sessions, case_ids=[first_case], offsets_in=[0])
results

In [ ]:
results = runner.run_sweep(sessions)
results

Optional: redo the strength checks of the runs still in `outputs/` without solving again. Use it after changing strengths in `materials.csv`, or for runs solved before the 3D interaction index existed. Those runs first read their stress fields from `model.mechdb` (a fresh Mechanical per run, about 20 s); runs open in Mechanical are skipped. `case_ids` limits it to some cases.

In [ ]:
results = runner.recheck()
results

Images for one run. To compare runs, open `results_viewer.ipynb`.

In [ ]:
case_id, offset_in = int(config.load_sweep()[0]["case_id"]), 0.0
folder = config.run_dir(case_id, offset_in * 25.4) / "images"
for name in ["contact_tool_pedal_box_pressure.png", "directional_deformation_z.png"]:
    if (folder / name).exists():
        print(name)
        display(Image(filename=str(folder / name), width=800))

In [ ]:
sessions.close()